In [13]:
import torch
from einops import rearrange, repeat
def create_rope_frequencies(seq_len, dim = 2, theta = 10000.0):
    assert dim % 2 == 0, "Dimension should be even numbers, most likely base of 2."
    common_freq = 1 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    position_mul = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(position_mul, common_freq) # 각 행은 임베딩 차원 절반 크기인 frequency vector.
    sin_emb = torch.sin(freqs)
    cos_emb = torch.cos(freqs)
    return sin_emb, cos_emb


In [5]:
def rotate_half_emb(x):
    x = rearrange(x, '... (d r) -> ... d r', r=2) # 이걸로 두개씩 묶어내기. .chunk도 가능할듯
    x1, x2 = x.unbind(dim=-1) # 두개씩 묶어낸 것들끼리 또 쪼갠다. 홀수 임베딩이랑 짝수 임베딩이랑 따로 분리됨
    x = torch.stack([-x2, x1], dim=-1) # 벡터로 sin, cos 연산하기 위해 필요한 과정.

    return rearrange(x, '... d r -> ... (d r)')

In [18]:
t = torch.randn(2, 3, 4)
print(t, "\n", rotate_half_emb(t))

tensor([[[ 0.3658, -1.7079,  0.3974,  0.8170],
         [ 1.3583, -0.1085,  0.5880, -1.0346],
         [-0.6927,  0.4395,  0.0372,  0.6916]],

        [[ 1.0244,  1.5291, -0.2270,  0.3424],
         [-0.3715, -1.0306,  0.5827, -0.4827],
         [-1.9580,  0.6376,  1.7415, -2.1386]]]) 
 tensor([[[ 1.7079,  0.3658, -0.8170,  0.3974],
         [ 0.1085,  1.3583,  1.0346,  0.5880],
         [-0.4395, -0.6927, -0.6916,  0.0372]],

        [[-1.5291,  1.0244, -0.3424, -0.2270],
         [ 1.0306, -0.3715,  0.4827,  0.5827],
         [-0.6376, -1.9580,  2.1386,  1.7415]]])


In [21]:
seq_len = t.size(1)
dimension = t.size(2)
sin_emb, cos_emb = create_rope_frequencies(seq_len, dimension)

print(sin_emb, "\n", cos_emb)

print(t * repeat(cos_emb, 'n d2 -> n (d2 r)', r=2) + rotate_half_emb(t) * repeat(sin_emb, 'n d2 -> n (d2 r)', r=2))


tensor([[0.0000, 0.0000],
        [0.8415, 0.0100],
        [0.9093, 0.0200]]) 
 tensor([[ 1.0000,  1.0000],
        [ 0.5403,  0.9999],
        [-0.4161,  0.9998]])
tensor([[[ 0.3658, -1.7079,  0.3974,  0.8170],
         [ 0.8252,  1.0843,  0.5983, -1.0286],
         [-0.1114, -0.8127,  0.0233,  0.6922]],

        [[ 1.0244,  1.5291, -0.2270,  0.3424],
         [ 0.6665, -0.8695,  0.5875, -0.4769],
         [ 0.2351, -2.0457,  1.7839, -2.1034]]])
